# Module 0 — Load Files into Tables (gold build + entity resolution)

Builds the **gold** Delta tables **directly from the CSVs** you uploaded under **Files**, with the correct data types, and resolves the payments customer + lending borrower into **one master `Customer`** (keyed by email).

> No bronze/silver layers — only **gold** is consumed by the semantic models and the ontology, so this keeps the build simple and avoids schema-merge issues. Types are set explicitly (IDs = text, codes/keys = bigint, money = double, dates = timestamp, flags = boolean).

**Before running:**
1. Attach this notebook to your lakehouse — left **Explorer** → **Add** → your lakehouse (the one holding the uploaded Files).
2. Check the `BASE` path in the next cell matches where you uploaded the files (here: `Files/NorthwindLakehouse/{payments,lending,truth}`).
3. **Run all**.

**Success looks like:** `gold.dim_customer` = **2300** rows, **600** in both lines of business, and the cross-domain check returns **13**.


In [ ]:
from pyspark.sql import functions as F, Window

# Files subfolder where you uploaded the CSVs. Change this if your folder names differ.
BASE = "Files/NorthwindLakehouse"

def read_csv(rel):
    return (spark.read.option("header", True).option("inferSchema", True)
            .csv(f"{BASE}/{rel}"))

# Sanity check — list what was uploaded so you can confirm the paths before loading.
for sub in ["payments", "lending", "truth"]:
    print(f"--- {BASE}/{sub} ---")
    for f in notebookutils.fs.ls(f"{BASE}/{sub}"):
        print("   ", f.name)


In [ ]:
# ---------------- BUILD GOLD (directly from the CSVs, with correct types) ----------------
for s in ["bronze", "silver", "gold"]:
    spark.sql(f"DROP SCHEMA IF EXISTS {s} CASCADE")
spark.sql("CREATE SCHEMA gold")

customers = read_csv("payments/customers.csv"); borrowers = read_csv("lending/borrowers.csv")
accounts = read_csv("payments/accounts.csv"); merchants = read_csv("payments/merchants.csv")
products = read_csv("payments/products.csv"); branches = read_csv("payments/branches.csv")
transactions = read_csv("payments/transactions.csv"); fraud = read_csv("payments/fraud_alerts.csv")
loans = read_csv("lending/loans.csv"); delinq = read_csv("lending/delinquency_events.csv")

# Entity resolution -> master Customer (unify payments + lending by email)
pay = customers.select(F.col("customerId").alias("paymentsCustomerId"), "name", "email", "segment", "region", "homeCountry", "riskScore")
bor = borrowers.select("borrowerId", "name", "email", "segment", "region", "homeCountry")
emails = pay.select("email").union(bor.select("email")).distinct()
w = Window.orderBy("email")
master = emails.withColumn("customerKey", F.concat(F.lit("CUST-"), F.lpad(F.row_number().over(w).cast("string"), 6, "0")))
dim_customer = (master
    .join(pay, "email", "left")
    .join(bor.select("email", F.col("name").alias("bn"), F.col("segment").alias("bs"), F.col("region").alias("br"), F.col("homeCountry").alias("bh"), "borrowerId"), "email", "left")
    .select("customerKey", "email",
            F.coalesce("name", "bn").alias("name"), F.coalesce("segment", "bs").alias("segment"),
            F.coalesce("region", "br").alias("region"), F.coalesce("homeCountry", "bh").alias("homeCountry"),
            F.col("riskScore").cast("double").alias("riskScore"), "paymentsCustomerId", "borrowerId",
            F.col("paymentsCustomerId").isNotNull().alias("hasDeposits"), F.col("borrowerId").isNotNull().alias("hasLoans")))

key_pay = dim_customer.where(F.col("paymentsCustomerId").isNotNull()).select("customerKey", F.col("paymentsCustomerId").alias("_p"))
key_bor = dim_customer.where(F.col("borrowerId").isNotNull()).select("customerKey", F.col("borrowerId").alias("_b"))

# Dimensions (explicit correct types)
dim_merchant = merchants.select("merchantId", "name", "category", F.col("mcc").cast("long").alias("mcc"), "country", F.col("riskFlag").cast("boolean").alias("riskFlag"))
dim_account = accounts.select("accountId", "customerId", "productId", "branchId", F.col("balance").cast("double").alias("balance"), "status", F.to_timestamp("openedDate").alias("openedDate"))
dim_product = products.select("productId", "name", "family")
dim_branch = branches.select("branchId", "name", "region")

# Facts (typed) with master customerKey
tx = (transactions.withColumn("timestamp", F.to_timestamp("timestamp")).withColumn("amount", F.col("amount").cast("double"))
      .withColumn("isFraud", F.col("isFraud").cast("boolean")).withColumn("mcc", F.col("mcc").cast("long")))
fact_transaction = tx.join(key_pay, F.col("customerId") == F.col("_p"), "left").drop("_p").withColumn("dateKey", F.date_format("timestamp", "yyyyMMdd").cast("long"))
fact_fraud_alert = fraud.withColumn("raisedAt", F.to_timestamp("raisedAt")).join(key_pay, F.col("customerId") == F.col("_p"), "left").drop("_p")
ln = (loans.withColumn("principal", F.col("principal").cast("double")).withColumn("rate", F.col("rate").cast("double"))
      .withColumn("termMonths", F.col("termMonths").cast("long")).withColumn("balance", F.col("balance").cast("double"))
      .withColumn("dpd", F.col("dpd").cast("long")).withColumn("originationDate", F.to_timestamp("originationDate")))
fact_loan = ln.join(key_bor, F.col("borrowerId") == F.col("_b"), "left").drop("_b")
fact_delinquency = (delinq.withColumn("flaggedAt", F.to_timestamp("flaggedAt"))
    .join(loans.select("loanId", "borrowerId"), "loanId", "left")
    .join(key_bor, F.col("borrowerId") == F.col("_b"), "left").drop("_b"))

d = tx.select(F.to_date("timestamp").alias("d")).distinct()
dim_date = d.select(F.col("d").cast("timestamp").alias("date"), F.year("d").cast("long").alias("year"),
                    F.quarter("d").cast("long").alias("quarter"), F.month("d").cast("long").alias("month"),
                    F.dayofmonth("d").cast("long").alias("day"), F.date_format("d", "yyyyMMdd").cast("long").alias("dateKey"))

for name, df in [("dim_customer", dim_customer), ("dim_merchant", dim_merchant), ("dim_account", dim_account),
                 ("dim_product", dim_product), ("dim_branch", dim_branch), ("dim_date", dim_date),
                 ("fact_transaction", fact_transaction), ("fact_fraud_alert", fact_fraud_alert),
                 ("fact_loan", fact_loan), ("fact_delinquency", fact_delinquency)]:
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"gold.{name}")
print("gold built (10 tables)")


In [ ]:
# ---------------- VERIFY ----------------
for t in ["gold.dim_customer", "gold.fact_transaction", "gold.fact_fraud_alert",
          "gold.fact_loan", "gold.fact_delinquency", "gold.dim_merchant"]:
    print(f"{t} = {spark.table(t).count()}")

res = dim_customer.agg(
    F.sum(F.col("hasDeposits").cast("int")).alias("with_deposits"),
    F.sum(F.col("hasLoans").cast("int")).alias("with_loans"),
    F.sum((F.col("hasDeposits") & F.col("hasLoans")).cast("int")).alias("both_lobs")).collect()[0]
print(f"customers: with_deposits={res.with_deposits} with_loans={res.with_loans} BOTH_LOBS={res.both_lobs}")

headline = spark.sql("""
  SELECT COUNT(DISTINCT c.customerKey) AS delinquent_borrowers_with_fraud
  FROM gold.dim_customer c
  JOIN (SELECT DISTINCT customerKey FROM gold.fact_delinquency WHERE customerKey IS NOT NULL) d ON c.customerKey=d.customerKey
  JOIN (SELECT DISTINCT customerKey FROM gold.fact_fraud_alert  WHERE customerKey IS NOT NULL) f ON c.customerKey=f.customerKey
  JOIN (SELECT DISTINCT customerKey FROM gold.fact_loan         WHERE customerKey IS NOT NULL) l ON c.customerKey=l.customerKey
""").collect()[0][0]
print(f"\nHEADLINE cross-domain (delinquent borrowers with fraud alerts) = {headline}  (expect 13)")
